In [ ]:
"""
upi_simulator_datapump.py
A high-throughput Python simulator and datapump for synthetic UPI/eFRM transaction events.
Publishes realistic synthetic transactions to a Kafka topic at a controlled TPS.

Usage:
    python upi_simulator_datapump.py --time 60 --threads 8 --tps 1000 --topic upi-events --bootstrap-servers localhost:9092
"""

import argparse
import json
import random
import time
import threading
import uuid
from datetime import datetime, timezone
from kafka import KafkaProducer

# ==========================================
# Synthetic Master Data (Expanded)
# ==========================================
APPS = [
    "com.phonepe.app", "net.one97.paytm", "in.org.npci.upiapp",
    "com.google.android.apps.nbu.paisa.user", "in.amazon.mShop.android.shopping",
    "com.mobikwik_new", "com.freecharge.android", "com.csam.icici.bank.imobile", 
    "com.axis.mobile", "com.sbi.upi", "com.dbs.in.digitalbank"
]
APP_NAMES = [
    "PhonePe", "Paytm", "BHIM", "Google Pay", "Amazon Pay",
    "MobiKwik", "Freecharge", "iMobile", "Axis Mobile", "YONO", "DBS digibank"
]
OS_LIST = ["Android", "iOS"]
ANDROID_VERSIONS = ["11", "12", "13", "14", "15", "16"]
IOS_VERSIONS = ["16", "17", "18", "19"]
DEVICE_TYPES = ["ANDROID_PHONE", "IOS_PHONE", "ANDROID_TABLET", "IOS_TABLET"]
DEVICE_BRANDS = [
    "SAMSUNG", "APPLE", "ONEPLUS", "XIAOMI", "REALME", "VIVO",
    "OPPO", "MOTOROLA", "GOOGLE", "NOTHING", "NOKIA", "POCO"
]
DEVICE_MODELS = [
    "Galaxy A15", "Galaxy A35", "Galaxy A55", "Galaxy S23", "Galaxy S24",
    "iPhone 14", "iPhone 15", "iPhone 16", "OnePlus 11", "OnePlus 12",
    "Redmi Note 13", "Redmi Note 14", "Realme 12", "Vivo V30",
    "Oppo Reno 11", "Moto G84", "Pixel 8", "Pixel 9", "Nothing Phone 2"
]
BANKS = [
    ("HDFC", "HDFC Bank"), ("ICIC", "ICICI Bank"), ("SBIN", "State Bank of India"),
    ("UTIB", "Axis Bank"), ("KKBK", "Kotak Mahindra Bank"), ("YESB", "YES BANK"),
    ("PUNB", "Punjab National Bank"), ("CNRB", "Canara Bank"), ("BARB", "Bank of Baroda"), 
    ("BKID", "Bank of India"), ("UBIN", "Union Bank of India"), ("IDIB", "Indian Bank"),
    ("IOBA", "Indian Overseas Bank"), ("FDRL", "Federal Bank"), ("INDB", "IndusInd Bank"), 
    ("IDFB", "IDFC FIRST Bank"), ("AUBL", "AU Small Finance Bank"), ("RBLB", "RBL Bank"),
    ("JIOP", "Jio Payments Bank")
]
BANK_TYPES = ["PUBLIC", "PRIVATE", "SMALL_FINANCE", "PAYMENTS", "FOREIGN", "COOPERATIVE"]
HANDLES = [
    "@ybl", "@axl", "@okhdfcbank", "@oksbi", "@okicici", "@paytm",
    "@ibl", "@upi", "@okaxis", "@okyes", "@okbob", "@okcanara", "@okunion"
]
PSPS = [
    "PHONEPE", "GOOGLEPAY", "PAYTM", "AMAZONPAY", "BHARATPE",
    "CRED", "MOBIKWIK", "NAVI", "BANK_PSP", "OTHER_TPAP"
]
PAYMENT_CHANNELS = ["UPI_APP", "QR", "INTENT", "COLLECT", "IN_APP", "WEB", "BANK_APP"]
TRANSACTION_TYPES = ["P2P", "P2M", "P2A", "COLLECT", "REFUND", "REVERSAL", "MANDATE"]
PAYMENT_FLOWS = ["PAY", "COLLECT", "REFUND", "REVERSAL", "MANDATE"]
INTEGRATION_MODES = ["QR", "INTENT", "COLLECT", "APP"]
PURPOSE_CODES = ["00", "01", "02"]
INITIATION_MODES = ["00", "01", "02", "11"]
CURRENCIES = ["INR"]
COUNTRIES = ["IN"]
TELECOM_OPERATORS = ["JIO", "AIRTEL", "VI", "BSNL"]
NETWORK_TYPES = ["4G", "5G", "WIFI", "3G"]
ZONES = ["EAST", "WEST", "NORTH", "SOUTH", "CENTRAL", "NORTHEAST"]
STATES = [
    "WEST_BENGAL", "MAHARASHTRA", "KARNATAKA", "TAMIL_NADU", "TELANGANA", "DELHI", 
    "GUJARAT", "RAJASTHAN", "UTTAR_PRADESH", "BIHAR", "ODISHA", "ASSAM", "KERALA", 
    "PUNJAB", "HARYANA", "MADHYA_PRADESH", "ANDHRA_PRADESH", "JHARKHAND", 
    "CHHATTISGARH", "UTTARAKHAND"
]
CITIES = [
    "KOLKATA", "MUMBAI", "BENGALURU", "CHENNAI", "HYDERABAD", "NEW_DELHI", "PUNE", 
    "AHMEDABAD", "JAIPUR", "BHUBANESWAR", "LUCKNOW", "PATNA", "GUWAHATI", "KOCHI", 
    "INDORE", "SURAT"
]
CUSTOMER_TYPES = ["INDIVIDUAL", "MERCHANT", "CORPORATE", "SMALL_BUSINESS"]
ACCOUNT_TYPES = ["SAVINGS", "CURRENT", "SALARY", "NRE", "NRO"]
MERCHANT_TYPES = [
    "GROCERY", "RESTAURANT", "FUEL", "E_COMMERCE", "TRAVEL", "EDUCATION", 
    "HEALTHCARE", "UTILITY", "RETAIL", "ENTERTAINMENT", "PHARMACY", "FOOD_DELIVERY", 
    "TRANSPORT", "HOTEL"
]
MERCHANT_SIZES = ["MICRO", "SMALL", "MEDIUM", "LARGE", "ENTERPRISE"]
MCC_CODES = [
    "5411", "5812", "5541", "5311", "5912", "4111", "4722", "5732", 
    "6012", "6300", "7011", "7999", "4900", "8299"
]
AUTH_METHODS = ["UPI_PIN", "BIOMETRIC", "DEVICE_AUTH", "OTP"]
AUTH_STATUSES = ["SUCCESS", "FAILED", "NOT_REQUIRED"]
PROCESSING_STATUSES = ["SUCCESS", "FAILURE", "PENDING", "TIMEOUT", "REVERSED", "DEEMED"]
DECISIONS = ["ALLOW", "DECLINE", "CHALLENGE", "REVIEW"]
RISK_LEVELS = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]
FRAUD_TYPES = [
    "NONE", "ACCOUNT_TAKEOVER", "SOCIAL_ENGINEERING", "MULE_ACCOUNT",
    "DEVICE_COMPROMISE", "VELOCITY_ABUSE", "SUSPICIOUS_MERCHANT",
    "SUSPICIOUS_BENEFICIARY", "LOCATION_ANOMALY"
]
RESULT_ERR_CODES = [
    ("SUCCESS", "00"), ("FAILURE", "U30"), ("FAILURE", "FR"), ("FAILURE", "U01"),
    ("FAILURE", "U16"), ("FAILURE", "U17"), ("FAILURE", "U28"), ("PENDING", "91"),
]
SERVER_REGIONS = ["EAST", "WEST", "NORTH", "SOUTH"]
SERVICE_NAMES = [
    "upi-gateway", "upi-switch", "risk-engine", "datalake-consumer",
    "notification-service", "case-management-service"
]
PROCESSOR_TYPES = ["ONLINE", "OFFLINE"]
LANGUAGES = ["en_IN", "bn_IN", "hi_IN", "ta_IN", "te_IN", "mr_IN", "gu_IN"]
TIME_PERIODS = ["EARLY_MORNING", "MORNING", "AFTERNOON", "EVENING", "NIGHT", "LATE_NIGHT"]

# ==========================================
# Event Generator
# ==========================================
def generate_upi_event():
    """Generates a rich, synthetic UPI/eFRM transaction event."""
    txn_id = f"TXN{uuid.uuid4().hex[:14].upper()}"
    timestamp = datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")
    
    # Log-normal distribution for realistic amount skew (mostly small, some large)
    amount = round(random.lognormvariate(4.5, 1.2), 2)
    amount = min(max(amount, 1.0), 200000.0)
    
    payer_id = f"P{random.randint(10000000, 99999999)}"
    payee_id = f"B{random.randint(10000000, 99999999)}"
    
    payer_vpa = f"{payer_id.lower()}{random.choice(HANDLES)}"
    payee_vpa = f"{payee_id.lower()}{random.choice(HANDLES)}"
    
    # Bank selection (unpacking tuple)
    payer_bank_code, payer_bank_name = random.choice(BANKS)
    payee_bank_code, payee_bank_name = random.choice(BANKS)
    
    # Device & Network
    device_os = random.choice(OS_LIST)
    device_os_version = random.choice(ANDROID_VERSIONS if device_os == "Android" else IOS_VERSIONS)
    
    # Transaction context
    txn_type = random.choice(TRANSACTION_TYPES)
    is_p2m = txn_type in ["P2M", "COLLECT"]
    
    # Merchant details (only if P2M/COLLECT)
    merchant_id = f"MER{random.randint(100000, 999999)}" if is_p2m else None
    merchant_type = random.choice(MERCHANT_TYPES) if is_p2m else None
    merchant_size = random.choice(MERCHANT_SIZES) if is_p2m else None
    mcc_code = random.choice(MCC_CODES) if is_p2m else None
    
    # Risk scoring (correlated with fraud)
    risk_score = round(random.uniform(0, 100), 2)
    if random.random() < 0.03:  # 3% chance of high risk spike
        risk_score = round(random.uniform(75, 100), 2)
        
    # Determine risk level
    if risk_score >= 75:
        risk_level = "CRITICAL"
    elif risk_score >= 50:
        risk_level = "HIGH"
    elif risk_score >= 25:
        risk_level = "MEDIUM"
    else:
        risk_level = "LOW"
        
    # Fraud logic
    is_fraud = 1 if risk_score > 85 and random.random() > 0.3 else 0
    fraud_type = "NONE"
    if is_fraud:
        fraud_type = random.choice(["ACCOUNT_TAKEOVER", "SOCIAL_ENGINEERING", "VELOCITY_ABUSE", "LOCATION_ANOMALY", "MULE_ACCOUNT"])
    
    # Processing status & Response code (correlated with risk)
    if risk_score >= 75 and random.random() > 0.4:
        proc_status, resp_code = "FAILURE", "U30"
        decision = "DECLINE"
    else:
        proc_status, resp_code = random.choice(RESULT_ERR_CODES)
        decision = "REVIEW" if risk_score >= 50 else "CHALLENGE" if risk_score >= 30 else "ALLOW"

    event = {
        "txn_id": txn_id,
        "timestamp": timestamp,
        "payer_id": payer_id,
        "payer_vpa": payer_vpa,
        "payer_bank_code": payer_bank_code,
        "payer_bank_name": payer_bank_name,
        "payer_customer_type": random.choice(CUSTOMER_TYPES),
        "payer_account_type": random.choice(ACCOUNT_TYPES),
        "payee_id": payee_id,
        "payee_vpa": payee_vpa,
        "payee_bank_code": payee_bank_code,
        "payee_bank_name": payee_bank_name,
        "amount": amount,
        "currency": random.choice(CURRENCIES),
        "transaction_type": txn_type,
        "payment_flow": random.choice(PAYMENT_FLOWS),
        "payment_channel": random.choice(PAYMENT_CHANNELS),
        "integration_mode": random.choice(INTEGRATION_MODES),
        "purpose_code": random.choice(PURPOSE_CODES),
        "initiation_mode": random.choice(INITIATION_MODES),
        "app": random.choice(APPS),
        "app_name": random.choice(APP_NAMES),
        "device_id": uuid.uuid4().hex,
        "device_type": random.choice(DEVICE_TYPES),
        "device_brand": random.choice(DEVICE_BRANDS),
        "device_model": random.choice(DEVICE_MODELS),
        "device_os": device_os,
        "device_os_version": device_os_version,
        "device_language": random.choice(LANGUAGES),
        "mobile_operator": random.choice(TELECOM_OPERATORS),
        "network_type": random.choice(NETWORK_TYPES),
        "ip_address": f"{random.randint(10,250)}.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,254)}",
        "zone": random.choice(ZONES),
        "state": random.choice(STATES),
        "city": random.choice(CITIES),
        "country": random.choice(COUNTRIES),
        "merchant_id": merchant_id,
        "merchant_type": merchant_type,
        "merchant_size": merchant_size,
        "merchant_mcc": mcc_code,
        "auth_method": random.choice(AUTH_METHODS),
        "auth_status": random.choice(AUTH_STATUSES),
        "risk_score": risk_score,
        "risk_level": risk_level,
        "fraud_type": fraud_type,
        "is_fraud": is_fraud,
        "processing_status": proc_status,
        "response_code": resp_code,
        "decision": decision,
        "processor_type": random.choice(PROCESSOR_TYPES),
        "server_region": random.choice(SERVER_REGIONS),
        "service_name": random.choice(SERVICE_NAMES),
        "processing_latency_ms": random.randint(5, 250),
        "time_period": random.choice(TIME_PERIODS)
    }
    return event

# ==========================================
# Worker Thread
# ==========================================
def worker_thread(thread_id, producer, topic, stop_event, start_time, duration, tps, num_threads, stats):
    """Worker function to generate and send events."""
    # Calculate sleep time per thread to maintain global TPS without lock contention
    sleep_duration = num_threads / tps if tps > 0 else 0
    count = 0
    
    while not stop_event.is_set() and (time.monotonic() - start_time) < duration:
        loop_start = time.monotonic()
        
        # Generate event
        event = generate_upi_event()
        event['simulator_thread'] = thread_id
        
        # Send to Kafka (Asynchronous)
        try:
            producer.send(topic, value=event)
            count += 1
        except Exception as e:
            print(f"[Thread {thread_id}] Error sending to Kafka: {e}")
            
        # Calculate remaining sleep time to maintain exact TPS
        elapsed = time.monotonic() - loop_start
        sleep_time = sleep_duration - elapsed
        if sleep_time > 0:
            time.sleep(sleep_time)
            
    stats[thread_id] = count

# ==========================================
# Main Execution
# ==========================================
def main():
    parser = argparse.ArgumentParser(description="UPI/eFRM Synthetic Data Simulator & Kafka Datapump")
    parser.add_argument("-t", "--time", type=int, default=60, help="Duration to run the simulator in seconds (default: 60)")
    parser.add_argument("-th", "--threads", type=int, default=4, help="Number of concurrent threads (default: 4)")
    parser.add_argument("-r", "--tps", type=int, default=100, help="Target transactions per second (total across all threads, default: 100)")
    parser.add_argument("--topic", type=str, default="upi-efrm-events", help="Kafka topic to publish to (default: upi-efrm-events)")
    parser.add_argument("--bootstrap-servers", type=str, default="localhost:9092", help="Kafka bootstrap servers (default: localhost:9092)")
    
    args = parser.parse_args()
    
    print(f"Starting UPI Simulator...")
    print(f"Target TPS: {args.tps} | Threads: {args.threads} | Duration: {args.time}s")
    print(f"Kafka Topic: {args.topic} | Servers: {args.bootstrap-servers}")
    print("-" * 60)
    
    # Initialize Kafka Producer
    producer = KafkaProducer(
        bootstrap_servers=args.bootstrap_servers.split(','),
        value_serializer=lambda m: json.dumps(m).encode('utf-8'),
        acks='1',
        linger_ms=5,          # Batch slightly for better throughput
        batch_size=32768,     # Increased batch size for high TPS
        compression_type='snappy'
    )
    
    stop_event = threading.Event()
    stats = {}
    threads = []
    start_time = time.monotonic()
    
    # Start worker threads
    for i in range(args.threads):
        t = threading.Thread(
            target=worker_thread, 
            args=(i, producer, args.topic, stop_event, start_time, args.time, args.tps, args.threads, stats)
        )
        t.start()
        threads.append(t)
        
    # Monitor progress
    try:
        while time.monotonic() - start_time < args.time:
            time.sleep(1)
            elapsed = time.monotonic() - start_time
            total_sent = sum(stats.values())
            current_tps = total_sent / elapsed if elapsed > 0 else 0
            print(f"[{elapsed:.1f}s] Total Sent: {total_sent:,} | Current Avg TPS: {current_tps:.1f}")
    except KeyboardInterrupt:
        print("\nInterrupted by user. Stopping...")
        stop_event.set()
        
    # Wait for all threads to finish
    for t in threads:
        t.join()
        
    # Flush and close producer
    producer.flush()
    producer.close()
    
    total_sent = sum(stats.values())
    actual_duration = time.monotonic() - start_time
    final_tps = total_sent / actual_duration if actual_duration > 0 else 0
    
    print("-" * 60)
    print(f"Simulation Complete!")
    print(f"Total Transactions: {total_sent:,}")
    print(f"Actual Duration: {actual_duration:.2f}s")
    print(f"Average TPS: {final_tps:.1f}")

if __name__ == "__main__":
    main()